# TDA-Net quick demo

Load GODAS input and monthly statistics, run trained TDA-Net weights, and save/plot the SST forecast.
The default predicts **January 2022 from December 2021 with member1**. The loading → model → prediction → postprocessing workflow follows the [ORCA-DL demo](https://github.com/OpenEarthLab/ORCA-DL/blob/main/demo.ipynb), adapted for TDA-Net's month input and released member weights.

Install the training/inference environment in [README.md](README.md), then download:
- [Weights](https://huggingface.co/HouZhongYi/TDA-Net): `checkpoints/member1.bin` (all ten for the ensemble).
- [Demo data and statistics](https://huggingface.co/datasets/HouZhongYi/TDA-Net-data): `example_data/GODAS/`, `stat/` and `resources/`.

Place these folders in the repository root and run the cells in order. The provided `.npy` inputs are **already normalized**; do not normalize them again. Statistics are needed to restore physical units. On the tested CPU, the single-member model takes roughly 13 seconds to load and 2 seconds for one prediction.

In [ ]:
from pathlib import Path
import gc
import json
import sys

import numpy as np
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd()
if not (ROOT / "model_config.json").is_file():
    raise FileNotFoundError("Open this notebook from the TDA-Net repository root.")
sys.path.insert(0, str(ROOT))
from model import ORCADLConfig, ORCADLModel

torch.set_num_threads(4)
torch.manual_seed(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 1. Choose the initialization and members

Use `MEMBERS = list(range(1, 11))` for the ten-member ensemble; weights are loaded one at a time. Set `STEPS = 6` for a six-month forecast. For a batch, add initialization months to `INIT_TIMES` and provide the eight corresponding files for each month. Each member ID is a release label, not a random seed.

In [ ]:
INIT_TIMES = ["2021_12"]
STEPS = 1
MEMBERS = [1]
DATA_DIR = ROOT / "example_data/GODAS"
STAT_DIR = ROOT / "stat"
CHECKPOINT_DIR = ROOT / "checkpoints"
OUTPUT_DIR = ROOT / "output/notebook_demo"

if STEPS < 1 or not MEMBERS or len(MEMBERS) != len(set(MEMBERS)):
    raise ValueError("Use positive STEPS and a nonempty list of unique members.")
if any(member not in range(1, 11) for member in MEMBERS):
    raise ValueError("Available member IDs are 1–10.")
initializations = [tuple(map(int, value.split("_"))) for value in INIT_TIMES]
if not initializations or any(not 1 <= month <= 12 for year, month in initializations):
    raise ValueError("INIT_TIMES must contain valid YYYY_M months.")

config = json.loads((ROOT / f"configs/member{MEMBERS[0]}.json").read_text())
variables = config["var_list"]
channels = config["in_chans"]
grid_shape = tuple(config["input_shape"])
stat_names = ["salt", "pottmp", "sst", "ucur", "vcur", "sshg", "uflx", "vflx"]
# Units: g/kg, °C, °C, m/s, m/s, m, N/m², N/m².
stats = {
    kind: {name: np.load(STAT_DIR / kind / f"{name}.npy", allow_pickle=False)
           for name in stat_names}
    for kind in ("mean", "std")
}

## 2. Load normalized input

The ocean variable order is `so, thetao, tos, uo, vo, zos`, with 16, 16, 1, 16, 16 and 1 channels. The atmosphere uses `tauu, tauv`. TDA-Net also receives each initialization's calendar month (1–12).

For your own GODAS NetCDF files, first regrid, convert units and normalize using the **initialization month**, as described under *GODAS preprocessing* in the README. The monthly normalization is `(field - mean[month - 1]) / std[month - 1]`.

In [ ]:
def load_input(variable, count, year, month):
    path = DATA_DIR / variable / f"{year}_{month}.npy"
    value = np.load(path, allow_pickle=False)
    if count == 1 and value.shape == grid_shape:
        value = value[None]
    if value.shape != (count,) + grid_shape or not np.isfinite(value).all():
        raise ValueError(f"Invalid normalized input: {path}, shape={value.shape}")
    return value.astype(np.float32, copy=False)

assert variables == ["so", "thetao", "tos", "uo", "vo", "zos"]
ocean_array = np.stack([
    np.concatenate([load_input(v, c, year, month) for v, c in zip(variables, channels)])
    for year, month in initializations
])
atmo_array = np.stack([
    np.concatenate([load_input(v, 1, year, month) for v in ("tauu", "tauv")])
    for year, month in initializations
])
ocean_vars = torch.from_numpy(ocean_array).to(device)
atmo_vars = torch.from_numpy(atmo_array).to(device)
month_tensor = torch.tensor([m for y, m in initializations], dtype=torch.long, device=device)
print("Ocean:", tuple(ocean_vars.shape), "Atmosphere:", tuple(atmo_vars.shape))

## 3. Load the model and predict

The selected members are averaged equally. Single-step and multi-step outputs are represented below with the same shape: `(batch, lead, 66, 128, 360)`.

In [ ]:
prediction_sum = None
for member in MEMBERS:
    checkpoint = CHECKPOINT_DIR / f"member{member}.bin"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Download the matching weight first: {checkpoint}")
    model = ORCADLModel(ORCADLConfig.from_json_file(str(ROOT / f"configs/member{member}.json")))
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    del state
    model.to(device).eval()
    with torch.inference_mode():
        output = model(ocean_vars=ocean_vars, atmo_vars=atmo_vars,
                       month=month_tensor, predict_time_steps=STEPS, return_dict=True)
        values = output.preds.detach().cpu().numpy()
    if STEPS == 1:
        values = values[:, None]
    expected = (len(INIT_TIMES), STEPS, sum(config["out_chans"])) + grid_shape
    if values.shape != expected or not np.isfinite(values).all():
        raise ValueError(f"Invalid prediction for member{member}: {values.shape}")
    if prediction_sum is None:
        prediction_sum = np.zeros(values.shape, dtype=np.float64)
    prediction_sum += values.astype(np.float64) / len(MEMBERS)
    print(f"Finished member{member}: {values.shape}")
    del model, output, values
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
predictions = prediction_sum.astype(np.float32)
del prediction_sum

## 4. Restore units and save

Use the **forecast target month** for inverse normalization, wrapping correctly from December to January. `pred_physical` contains all six ocean fields; the saved SST arrays have shape `(batch, lead, 128, 360)`.

In [ ]:
target_months = [
    [divmod(year * 12 + month - 1 + lead, 12) for lead in range(1, STEPS + 1)]
    for year, month in initializations
]
target_labels = [[f"{year}_{month0 + 1}" for year, month0 in row] for row in target_months]
blocks = np.split(predictions, np.cumsum(config["out_chans"])[:-1], axis=2)
pred_normalized = dict(zip(variables, blocks))
pred_physical = {}
for variable, stat_name in zip(variables, stat_names[:6]):
    values = pred_normalized[variable]
    pred_physical[variable] = np.stack([
        np.stack([values[b, lead] * stats["std"][stat_name][month0]
                  + stats["mean"][stat_name][month0]
                  for lead, (year, month0) in enumerate(row)])
        for b, row in enumerate(target_months)
    ])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.save(OUTPUT_DIR / "tos_normalized.npy", pred_normalized["tos"][:, :, 0])
np.save(OUTPUT_DIR / "tos_celsius.npy", pred_physical["tos"][:, :, 0])
summary = {"initializations": INIT_TIMES, "target_months": target_labels,
           "members": [f"member{i}" for i in MEMBERS], "steps": STEPS,
           "sst_shape": list(pred_physical["tos"][:, :, 0].shape), "device": device}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))
print("Saved:", OUTPUT_DIR)

## 5. Plot SST

The default view compares the first initialization with its first forecast month. Change `batch_index` and `lead_index` to view other cases or forecast leads.

In [ ]:
batch_index, lead_index = 0, 0
init_year, init_month = initializations[batch_index]
sst_channel = sum(channels[:variables.index("tos")])
initial_sst = (ocean_array[batch_index, sst_channel] * stats["std"]["sst"][init_month - 1]
               + stats["mean"]["sst"][init_month - 1])
forecast_sst = pred_physical["tos"][batch_index, lead_index, 0]
fields = [initial_sst, forecast_sst]
vmin, vmax = np.nanpercentile(np.stack(fields), [2, 98])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), constrained_layout=True)
for ax, field, title in zip(axes, fields, [
    f"GODAS initialization: {INIT_TIMES[batch_index]}",
    f"TDA-Net forecast: {target_labels[batch_index][lead_index]}"
]):
    im = ax.imshow(field, origin="lower", extent=(0, 360, -64, 64),
                   cmap="coolwarm", vmin=vmin, vmax=vmax, aspect="auto")
    ax.set(title=title, xlabel="Longitude (°E)", ylabel="Latitude (°)")
fig.colorbar(im, ax=axes, label="SST (°C)", shrink=0.85)
fig.savefig(OUTPUT_DIR / "sst_demo.png", dpi=150)
plt.show()